# Deepfake Detector — 3-Model Ensemble (Colab Backend)
**Models used:**
1. Your custom trained PyTorch model (.pt/.pth)
2. umm-maybe/AI-image-detector (HuggingFace)
3. Organika/sdxl-detector (HuggingFace)

Steps:
1. Cell 1 — install deps + paste ngrok token
2. Cell 2 — upload your .pt/.pth model
3. Cell 3 — EDIT to match your architecture, then run
4. Cell 4 — load HuggingFace models
5. Cell 5 — ensemble detection helpers
6. Cell 6 — start Flask + ngrok, copy the URL

> Enable GPU: Runtime > Change runtime type > T4 GPU

In [ ]:
# CELL 1 — Install dependencies
!pip install -q flask flask-cors pyngrok torch torchvision transformers opencv-python-headless Pillow

# Get free token at https://ngrok.com -> Your Authtoken
NGROK_TOKEN = ""  # <-- paste your token

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print("Done")

Done


In [2]:
# CELL 2 — Upload your trained model
from google.colab import files
import os

os.makedirs("uploads", exist_ok=True)
os.makedirs("frames",  exist_ok=True)

print("Upload your .pt or .pth file...")
uploaded = files.upload()
MODEL_PATH = list(uploaded.keys())[0]
print(f"Model uploaded: {MODEL_PATH}")

Upload your .pt or .pth file...


Saving deepfake_model.pth to deepfake_model.pth
Model uploaded: deepfake_model.pth


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as tv_models
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

custom_model = tv_models.resnet50(weights=None)
custom_model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(custom_model.fc.in_features, 2)
)

checkpoint = torch.load(MODEL_PATH, map_location=device)
if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint:
        custom_model.load_state_dict(checkpoint["model_state_dict"])
    elif "state_dict" in checkpoint:
        custom_model.load_state_dict(checkpoint["state_dict"])
    else:
        custom_model.load_state_dict(checkpoint)
else:
    custom_model = checkpoint

custom_model.to(device).eval()

CUSTOM_CLASS_NAMES = ["AI GENERATED", "REAL IMAGE"]
THRESHOLD = 0.75

custom_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print("Custom ResNet50 loaded successfully")
print(f"Classes: {CUSTOM_CLASS_NAMES}")
print(f"Best validation accuracy during training: 97.61%")

Device: cuda
Custom ResNet50 loaded successfully
Classes: ['AI GENERATED', 'REAL IMAGE']
Best validation accuracy during training: 97.61%


In [8]:
# CELL 4 — Load HuggingFace models
from transformers import AutoImageProcessor, AutoModelForImageClassification

print("Loading HF Model 1: umm-maybe/AI-image-detector ...")
hf_processor1 = AutoImageProcessor.from_pretrained("umm-maybe/AI-image-detector")
hf_model1 = AutoModelForImageClassification.from_pretrained("umm-maybe/AI-image-detector").to(device).eval()
print("HF Model 1 ready")

print("Loading HF Model 2: Organika/sdxl-detector ...")
hf_processor2 = AutoImageProcessor.from_pretrained("Organika/sdxl-detector")
hf_model2 = AutoModelForImageClassification.from_pretrained("Organika/sdxl-detector").to(device).eval()
print("HF Model 2 ready")

print("All 3 models loaded!")

Loading HF Model 1: umm-maybe/AI-image-detector ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/937 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/347M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/449 [00:00<?, ?it/s]

HF Model 2 ready
All 3 models loaded!


In [12]:
# CELL 5 — Ensemble detection helpers
import cv2, time, json

ANALYTICS_FILE = "analytics.json"

def load_analytics():
    default = {
        "AI GENERATED": 0, "REAL IMAGE": 0, "UNCERTAIN": 0,
        "LIKELY DEEPFAKE": 0, "LIKELY REAL": 0,
        "total_images": 0, "total_videos": 0
    }
    if not os.path.exists(ANALYTICS_FILE):
        return default
    with open(ANALYTICS_FILE) as f:
        return {**default, **json.load(f)}

def save_analytics(data):
    with open(ANALYTICS_FILE, "w") as f:
        json.dump(data, f, indent=4)

def log_result(verdict, kind="image"):
    data = load_analytics()
    data[f"total_{kind}s"] += 1
    if verdict in data:
        data[verdict] += 1
    save_analytics(data)


def predict_custom(image):
    tensor = custom_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(custom_model(tensor), dim=1)[0]
    ai_idx   = CUSTOM_CLASS_NAMES.index("AI GENERATED") if "AI GENERATED" in CUSTOM_CLASS_NAMES else 0
    real_idx = CUSTOM_CLASS_NAMES.index("REAL IMAGE")   if "REAL IMAGE"   in CUSTOM_CLASS_NAMES else 1
    return round(probs[ai_idx].item(), 4), round(probs[real_idx].item(), 4)


def predict_hf1(image):
    inputs = hf_processor1(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = F.softmax(hf_model1(**inputs).logits, dim=1)[0]
    return round(probs[0].item(), 4), round(probs[1].item(), 4)


def predict_hf2(image):
    inputs = hf_processor2(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = F.softmax(hf_model2(**inputs).logits, dim=1)[0]
    return round(probs[0].item(), 4), round(probs[1].item(), 4)


def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")

    c_ai,  c_real  = predict_custom(image)
    h1_ai, h1_real = predict_hf1(image)
    h2_ai, h2_real = predict_hf2(image)

    ens_ai   = round((c_ai   + h1_ai   + h2_ai)   / 3, 4)
    ens_real = round((c_real + h1_real + h2_real) / 3, 4)

    if ens_ai >= THRESHOLD:
        verdict, confidence = "AI GENERATED", ens_ai
    elif ens_real >= THRESHOLD:
        verdict, confidence = "REAL IMAGE", ens_real
    else:
        verdict, confidence = "UNCERTAIN", max(ens_ai, ens_real)

    per_model = {
        "Your Model":        {"ai_probability": c_ai,  "real_probability": c_real,  "verdict": "AI GENERATED" if c_ai  >= THRESHOLD else ("REAL IMAGE" if c_real  >= THRESHOLD else "UNCERTAIN")},
        "AI-Image-Detector": {"ai_probability": h1_ai, "real_probability": h1_real, "verdict": "AI GENERATED" if h1_ai >= THRESHOLD else ("REAL IMAGE" if h1_real >= THRESHOLD else "UNCERTAIN")},
        "SDXL-Detector":     {"ai_probability": h2_ai, "real_probability": h2_real, "verdict": "AI GENERATED" if h2_ai >= THRESHOLD else ("REAL IMAGE" if h2_real >= THRESHOLD else "UNCERTAIN")},
    }

    print(f"\n--- Results for {image_path} ---")
    for name, r in per_model.items():
        ai_val   = r["ai_probability"]
        real_val = r["real_probability"]
        verd     = r["verdict"]
        print(f"  {name:25s} | AI: {ai_val:.1%}  Real: {real_val:.1%}  -> {verd}")
    print(f"  {'ENSEMBLE':25s} | AI: {ens_ai:.1%}  Real: {ens_real:.1%}  -> {verdict} ({confidence:.1%})")
    print("-" * 65)

    return {
        "verdict":          verdict,
        "confidence":       round(confidence, 4),
        "ai_probability":   ens_ai,
        "real_probability": ens_real,
        "per_model":        per_model,
    }


def predict_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_count, ai_frames = 0, 0
    timeline = []
    per_model_sums = {
        "Your Model":        {"ai": 0, "real": 0},
        "AI-Image-Detector": {"ai": 0, "real": 0},
        "SDXL-Detector":     {"ai": 0, "real": 0},
    }
    frame_folder = f"frames/f_{int(time.time())}"
    os.makedirs(frame_folder, exist_ok=True)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1
        if frame_count % 30 != 0:
            continue
        fp = os.path.join(frame_folder, f"fr_{frame_count}.jpg")
        cv2.imwrite(fp, frame)
        result = predict_image(fp)
        ens_ai = result["ai_probability"]
        timeline.append(ens_ai)
        if ens_ai > 0.6:
            ai_frames += 1
        for name in per_model_sums:
            per_model_sums[name]["ai"]   += result["per_model"][name]["ai_probability"]
            per_model_sums[name]["real"] += result["per_model"][name]["real_probability"]

    cap.release()
    if not timeline:
        return {"error": "No frames extracted"}

    n        = len(timeline)
    ai_ratio = ai_frames / n
    verdict  = "LIKELY DEEPFAKE" if ai_ratio > 0.4 else "LIKELY REAL"

    per_model_avg = {
        name: {
            "avg_ai_probability":   round(per_model_sums[name]["ai"]   / n, 4),
            "avg_real_probability": round(per_model_sums[name]["real"] / n, 4),
        }
        for name in per_model_sums
    }

    print("\n--- Video Summary ---")
    for name, r in per_model_avg.items():
        avg_ai   = r["avg_ai_probability"]
        avg_real = r["avg_real_probability"]
        print(f"  {name:25s} | Avg AI: {avg_ai:.1%}  Avg Real: {avg_real:.1%}")
    print(f"  Ensemble AI frame ratio: {ai_ratio:.1%}  ->  {verdict}")
    print("-" * 65)

    return {
        "verdict":               verdict,
        "ai_frame_ratio":        round(ai_ratio, 4),
        "avg_ai_prob":           round(sum(timeline) / n, 4),
        "timeline":              timeline,
        "total_frames_sampled":  n,
        "per_model_avg":         per_model_avg,
    }

print("Ensemble helpers ready")

Ensemble helpers ready


In [13]:
# CELL 6 — Start Flask + ngrok
import threading, time as _t
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

@app.route("/health")
def health():
    return jsonify({
        "status": "ok",
        "device": str(device),
        "models": ["Your Model", "AI-Image-Detector", "SDXL-Detector"],
        "ensemble": "equal weight (33% each)"
    })

@app.route("/detect", methods=["POST"])
def detect_image_route():
    if "file" not in request.files:
        return jsonify({"error": "No file uploaded"}), 400
    f = request.files["file"]
    raw = f.read()
    if len(raw) > 10 * 1024 * 1024:
        return jsonify({"error": "Max 10 MB"}), 413
    path = os.path.join("uploads", f"{int(_t.time())}_{f.filename}")
    open(path, "wb").write(raw)
    try:
        result = predict_image(path)
        log_result(result["verdict"], "image")
        return jsonify(result)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/detect_video", methods=["POST"])
def detect_video_route():
    if "video" not in request.files:
        return jsonify({"error": "No video uploaded"}), 400
    f = request.files["video"]
    raw = f.read()
    if len(raw) > 100 * 1024 * 1024:
        return jsonify({"error": "Max 100 MB"}), 413
    path = os.path.join("uploads", f"{int(_t.time())}_{f.filename}")
    open(path, "wb").write(raw)
    try:
        result = predict_video(path)
        if "error" not in result:
            log_result(result["verdict"], "video")
        return jsonify(result)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/analytics")
def analytics_route():
    return jsonify(load_analytics())

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False, debug=False), daemon=True).start()
_t.sleep(2)

public_url = ngrok.connect(5000).public_url
print("=" * 60)
print(f"  API URL:  {public_url}")
print("=" * 60)
print("Paste this URL into your Render frontend.")
print("Keep this cell running to keep the API alive.")
print(f"  GET  {public_url}/health")
print(f"  POST {public_url}/detect        (field: file)")
print(f"  POST {public_url}/detect_video  (field: video)")
print(f"  GET  {public_url}/analytics")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


  API URL:  https://benzoic-suppletive-sarita.ngrok-free.dev
Paste this URL into your Render frontend.
Keep this cell running to keep the API alive.
  GET  https://benzoic-suppletive-sarita.ngrok-free.dev/health
  POST https://benzoic-suppletive-sarita.ngrok-free.dev/detect        (field: file)
  POST https://benzoic-suppletive-sarita.ngrok-free.dev/detect_video  (field: video)
  GET  https://benzoic-suppletive-sarita.ngrok-free.dev/analytics
